# Verify Processed Climate Data

This notebook verifies the processed NetCDF files by loading them, inspecting their metadata (Resolution, Bounding Box, Time Step, Units), and plotting the data.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Define paths
DATA_DIR = r'../data/processed'
FILE_PRECIP = os.path.join(DATA_DIR, 'precip_processed.nc')
FILE_TEMP = os.path.join(DATA_DIR, 'temp_processed.nc')
FILE_POP = os.path.join(DATA_DIR, 'pop_processed.nc')
FILE_SOIL = os.path.join(DATA_DIR, 'soil_processed.nc')

In [ ]:
def inspect_dataset(name, ds):
    print(f"--- Inspecting {name} ---\n")
    print(f"Variables: {list(ds.data_vars)}")
    
    # Grid Resolution & Bounding Box
    if 'lon' in ds.coords and 'lat' in ds.coords:
        lat = ds.lat
        lon = ds.lon
        
        lat_res = abs(lat[1] - lat[0]).values if len(lat) > 1 else np.nan
        lon_res = abs(lon[1] - lon[0]).values if len(lon) > 1 else np.nan
        
        print(f"Resolution: Lat {lat_res:.5f}, Lon {lon_res:.5f}")
        print(f"Bounding Box: Lon [{lon.min().values:.5f}, {lon.max().values:.5f}], Lat [{lat.min().values:.5f}, {lat.max().values:.5f}]")
    
    # Time info
    if 'time' in ds.coords:
        time = ds.time
        print(f"Time Duration: {time.min().values} to {time.max().values}")
        if len(time) > 1:
            diff = time[1] - time[0]
            if isinstance(diff.values, np.timedelta64):
                days = diff.values / np.timedelta64(1, 'D')
                print(f"Time Step: {days} days")
            else:
                print(f"Time Step: {diff.values}")
    
    # Units
    for var in ds.data_vars:
        units = ds[var].attrs.get('units', 'No units found')
        print(f"Variable '{var}' Units: {units}")
    print("\n")

## 1. Precipitation Data

In [ ]:
try:
    ds_pr = xr.open_dataset(FILE_PRECIP)
    inspect_dataset("Precipitation", ds_pr)
    
    # Plot first time step
    plt.figure(figsize=(10, 6))
    ds_pr['tp'].isel(time=0).plot()
    plt.title("Precipitation (First Time Step)")
    plt.show()
except Exception as e:
    print(f"Error loading Precip: {e}")

## 2. Temperature Data

In [ ]:
try:
    ds_t2 = xr.open_dataset(FILE_TEMP)
    inspect_dataset("Temperature", ds_t2)
    
    plt.figure(figsize=(10, 6))
    ds_t2['t2m'].isel(time=0).plot(cmap='coolwarm')
    plt.title("Temperature (First Time Step)")
    plt.show()
except Exception as e:
    print(f"Error loading Temp: {e}")

## 3. Population Data

In [ ]:
try:
    ds_pop = xr.open_dataset(FILE_POP)
    inspect_dataset("Population", ds_pop)
    
    plt.figure(figsize=(10, 6))
    ds_pop['population'].plot(cmap='viridis')
    plt.title("Population Density")
    plt.show()
except Exception as e:
    print(f"Error loading Pop: {e}")

## 4. Soil Data

In [ ]:
try:
    ds_soil = xr.open_dataset(FILE_SOIL)
    inspect_dataset("Soil", ds_soil)
    
    # Plot Clay
    if 'soilfraction_clay' in ds_soil:
        plt.figure(figsize=(10, 6))
        ds_soil['soilfraction_clay'].isel(time=0).plot(cmap='copper_r')
        plt.title("Soil Fraction: Clay (First Time Step)")
        plt.show()
except Exception as e:
    print(f"Error loading Soil: {e}")